# Colab 22 — Cross-representation summary: Accuracy & Retrieval in one place

The two cross-rep questions, previously scattered across colab15-21, answered side-by-side
on **one** 2x3 matrix ({AA, SS} encoders x {AA, SS, 3Di} feeds), with presentation-ready
charts.

> **Q1 - Accuracy.** Can embedding distance tell a high-similarity pair from a random one?
> Metric: **AUROC is-high** (positives = pairs with true score `>= 0.70`, negatives = the
> rest; discriminator = predicted L2-sim). One number per encoder x feed, 0.5 = chance,
> 1.0 = perfect.
>
> **Q2 - Retrieval.** Can it find a high-similarity neighbour in a crowded pool?
> Metric: **AA feed -> hits@10 %** (well-posed: one designated partner per query);
> **SS & 3Di feed -> frac@10 %** (crowded: a query has *many* genuine `>= 0.70`
> neighbours, so we score against the exact-Levenshtein neighbour SET). Reads as
> "encoder X retrieves Y% of the achievable high-sim neighbours in its top 10."

**Architecture = latest design (colab19 / colab17b recipe), frozen, NO calibrator.**
Two encoders: **AA** (20-letter, `BAND_LOW=0.30`) and **SS** (3-letter, `BAND_LOW=0.56`),
both K=16 AdaptiveAvgPool, 128-d L2-normalised, 3-bin CE head (discardable; retrieval and
AUROC use the encoder's L2 geometry only). RNG threaded for determinism.

Colours throughout: **AA-encoder = blue**, **SS-encoder = green**.

## 1. Setup

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')

In [ ]:
DATA_DIR = '/content/thesis-edit-distance-nn/sampledata/cath'
import os
for f in ['cath_s20_train70.csv.gz', 'cath_s20_test30.csv.gz', 'cath_eval.csv.gz', 'cath_s20_3di.csv.gz']:
    p = os.path.join(DATA_DIR, f)
    print(f'{"OK" if os.path.exists(p) else "MISSING":<8} {p}')

In [ ]:
!pip install torch torchvision rapidfuzz scikit-learn scipy --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from rapidfuzz.distance import Levenshtein as RFLevenshtein
from rapidfuzz.process import cdist as rf_cdist

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Presentation palette (consistent across both charts)
COL_AA_ENC = '#3b7dd8'   # blue  = AA-encoder
COL_SS_ENC = '#36a85a'   # green = SS-encoder

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Constants (encoder recipe identical to colab19 / colab17b)

In [ ]:
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'
SS_ALPHABET = 'HLS'
VOCAB_SIZE = len(AA_ALPHABET) + 1
PAD_IDX = len(AA_ALPHABET)
AA_SET = set(AA_ALPHABET)
SS_SET = set(SS_ALPHABET)
CHAR_TO_IDX = {c: i for i, c in enumerate(AA_ALPHABET)}

MIN_LEN = 50
MAX_LEN_SEQ = 200
MAX_LEN = 200

N_TRAIN = 30000
NUM_EPOCHS = 30
BATCH_SIZE = 128
K = 16

BAND_LOW_AA = 0.30
BAND_LOW_SS = 0.56
BAND_HIGH   = 0.70          # high-sim cutoff (positives for AUROC; oracle true-set bar)
BIN_NAMES = ['far', 'mid', 'high']

K_RETRIEVAL = 10            # @k for both hits@10 and frac@10
SEED_TRAIN_AA = 42
SEED_TRAIN_SS = 43

print(f'AA encoder BAND_LOW={BAND_LOW_AA} | SS encoder BAND_LOW={BAND_LOW_SS} '
      f'| high-sim cutoff={BAND_HIGH} | retrieval @k={K_RETRIEVAL}')

## 3. Helpers — Levenshtein, encode, perturb (RNG threaded for determinism)

In [ ]:
def norm_lev(a, b):
    L = max(len(a), len(b))
    return 1.0 if L == 0 else 1.0 - RFLevenshtein.distance(a, b) / L

def is_standard_aa(seq): return all(c in AA_SET for c in seq)
def is_standard_ss(seq): return all(c in SS_SET for c in seq)

def encode_pad(seq, max_len=MAX_LEN, pad_idx=PAD_IDX):
    idx = [CHAR_TO_IDX[c] for c in seq][:max_len]
    idx += [pad_idx] * (max_len - len(idx))
    return torch.tensor(idx, dtype=torch.long)

def perturb(seq, k, alphabet, rng, max_len=MAX_LEN):
    s = list(seq); abc = list(alphabet)
    for _ in range(k):
        if len(s) == 0: op = 'ins'
        elif len(s) >= max_len: op = rng.choice(['sub', 'del'])
        else: op = rng.choice(['sub', 'ins', 'del'])
        if op == 'sub':
            i = rng.integers(0, len(s)); choices = [c for c in abc if c != s[i]]; s[i] = rng.choice(choices)
        elif op == 'ins':
            i = rng.integers(0, len(s) + 1); s.insert(i, rng.choice(abc))
        elif op == 'del':
            i = rng.integers(0, len(s)); del s[i]
    return ''.join(s)

def random_seq(alphabet, rng, min_len=MIN_LEN, max_len=MAX_LEN_SEQ):
    length = int(rng.integers(min_len, max_len + 1))
    return ''.join(rng.choice(list(alphabet), size=length))

def bin_idx_for(x, band_low):
    if x < band_low: return 0
    if x < BAND_HIGH: return 1
    return 2

def make_full_range_pairs(alphabet, n_pairs, rng):
    """t ~ U[0,1] -> roughly uniform coverage of normLev (far/mid/high all present)."""
    pairs = []; attempts = 0; max_attempts = n_pairs * 4
    while len(pairs) < n_pairs and attempts < max_attempts:
        attempts += 1
        seed = random_seq(alphabet, rng)
        if len(seed) < MIN_LEN: continue
        L = len(seed); t = float(rng.uniform(0.0, 1.0)); k = max(0, int(round((1.0 - t) * L)))
        other = perturb(seed, k, alphabet, rng)
        if 1 <= len(other) <= MAX_LEN:
            pairs.append((seed, other, norm_lev(seed, other)))
    return pairs

## 4. Architecture + training (3-bin CE head, identical to colab19)

In [ ]:
class SiameseEncoder(nn.Module):
    def __init__(self, K, vocab_size=VOCAB_SIZE, embed_dim=32,
                 hidden1=32, hidden2=64, out_dim=128, pad_idx=PAD_IDX):
        super().__init__()
        self.K = K; self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.conv1 = nn.Conv1d(embed_dim, hidden1, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(hidden1, hidden2, kernel_size=3, padding=1)
        self.pool  = nn.AdaptiveAvgPool1d(K)
        self.fc    = nn.Linear(hidden2 * K, out_dim)

    def forward(self, x):
        mask = (x != self.pad_idx).float()
        e = self.embedding(x).permute(0, 2, 1)
        h = F.relu(self.conv1(e)); h = F.relu(self.conv2(h))
        h = h * mask.unsqueeze(1)
        h = self.pool(h).flatten(1)
        return F.normalize(self.fc(h), p=2, dim=1)


class SiameseClassifier(nn.Module):
    def __init__(self, K, embed_out=128, hidden_mlp=64, n_bins=3):
        super().__init__()
        self.encoder = SiameseEncoder(K)
        self.head = nn.Sequential(
            nn.Linear(embed_out, hidden_mlp), nn.LeakyReLU(0.01),
            nn.Linear(hidden_mlp, n_bins))
    def forward(self, a, b):
        return self.head(torch.abs(self.encoder(a) - self.encoder(b)))
    def encode(self, x):
        return self.encoder(x)


class PairDatasetCE(Dataset):
    def __init__(self, pairs, band_low):
        self.a = [encode_pad(a) for a, _, _ in pairs]
        self.b = [encode_pad(b) for _, b, _ in pairs]
        self.bins = torch.tensor([bin_idx_for(l, band_low) for _, _, l in pairs], dtype=torch.long)
    def __len__(self): return len(self.bins)
    def __getitem__(self, i): return self.a[i], self.b[i], self.bins[i]


def train_encoder(alphabet, band_low, label, train_seed, n_epochs=NUM_EPOCHS):
    torch.manual_seed(train_seed)
    rng = np.random.default_rng(train_seed)
    print(f'\n===== Training {label} encoder (alphabet={alphabet}, band_low={band_low}, seed={train_seed}) =====')
    pairs = make_full_range_pairs(alphabet, N_TRAIN, rng)
    labels = np.array([p[2] for p in pairs])
    counts = {b: int(c) for b, c in zip(BIN_NAMES,
              np.bincount([bin_idx_for(l, band_low) for l in labels], minlength=3))}
    print(f'  {len(pairs)} pairs, normLev in [{labels.min():.3f}, {labels.max():.3f}], bins={counts}')
    dl = DataLoader(PairDatasetCE(pairs, band_low), batch_size=BATCH_SIZE, shuffle=True)
    model = SiameseClassifier(K).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss()
    model.train()
    for epoch in range(1, n_epochs + 1):
        tot = 0.0; nb = 0
        for a, b, y in dl:
            a, b, y = a.to(device), b.to(device), y.to(device)
            loss = crit(model(a, b), y)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item(); nb += 1
        if epoch % 5 == 0 or epoch == 1:
            print(f'  [{label}] epoch {epoch:3d}/{n_epochs} CE={tot/nb:.4f}')
    model.eval()
    return model

## 5. Data load — pool, eval set, 3Di per-pair score (from colab19)

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_train70.csv.gz'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_test30.csv.gz'))
prot_df = pd.concat([train_df, test_df], ignore_index=True).drop_duplicates('domain_id')
prot_df = prot_df[prot_df['aa_seq'].apply(is_standard_aa)]
prot_df = prot_df[prot_df['ss_seq'].apply(is_standard_ss)]
prot_df = prot_df[prot_df['aa_seq'].str.len() == prot_df['ss_seq'].str.len()]
prot_df['aa_len'] = prot_df['aa_seq'].str.len()
RESCUED = {'4z0mC02', '3qkaE02'}
in_range = prot_df['aa_len'].between(MIN_LEN, MAX_LEN)
prot_df = prot_df[in_range | prot_df['domain_id'].isin(RESCUED)].reset_index(drop=True)

id_to_aa = dict(zip(prot_df['domain_id'], prot_df['aa_seq']))
id_to_ss = dict(zip(prot_df['domain_id'], prot_df['ss_seq']))
all_domains = list(prot_df['domain_id'])
print(f'Protein pool: {len(prot_df)}')

eval_df = pd.read_csv(os.path.join(DATA_DIR, 'cath_eval.csv.gz'))
seqs3 = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_3di.csv.gz'))
id_to_3di = dict(zip(seqs3['domain_id'], seqs3['3di'].astype(str)))

def pair_3di_score(a, b):
    if a in id_to_3di and b in id_to_3di:
        return norm_lev(id_to_3di[a], id_to_3di[b])
    return np.nan
eval_df['3di_score'] = [pair_3di_score(a, b) for a, b in zip(eval_df['domain_a'], eval_df['domain_b'])]

SCORE_COL = {'AA': 'aa_score', 'SS': 'ss_score', '3Di': '3di_score'}
LOOKUP    = {'AA': id_to_aa,  'SS': id_to_ss,  '3Di': id_to_3di}
for feed, sc in SCORE_COL.items():
    n = int((eval_df[sc] >= BAND_HIGH).sum())
    tag = 'powered' if n >= 200 else ('moderate' if n >= 30 else 'underpowered (n<30)')
    print(f'  {feed}-feed: {n} eval pairs >= {BAND_HIGH}  [{tag}]')

## 6. Train the two encoders (frozen afterwards)

In [ ]:
model_aa = train_encoder(AA_ALPHABET, BAND_LOW_AA, 'AA', SEED_TRAIN_AA)
model_ss = train_encoder(SS_ALPHABET, BAND_LOW_SS, 'SS', SEED_TRAIN_SS)

## 7. Metric machinery — predicted L2-sim, AUROC, eval-pair selection

- `pred_sim_strings`: deployment score `1 - ||e_a - e_b|| / 2` for arbitrary pairs.
- `encode_pool`: batch-encode the full retrieval pool once per (encoder, feed).
- `auroc_cell`: **Q1** number. is-high AUROC over the full eval set of a feed
  (positives = `>= 0.70`, negatives = the rest), discriminator = predicted L2-sim.
- `feed_high_pairs`: the `>= 0.70` eval pairs with both proteins in pool (the retrieval queries).

In [ ]:
@torch.no_grad()
def encode_pool(model, seq_lookup, ids, batch_size=256):
    model.eval(); valid = [d for d in ids if d in seq_lookup]; out = []
    for i in range(0, len(valid), batch_size):
        x = torch.stack([encode_pad(seq_lookup[d]) for d in valid[i:i+batch_size]]).to(device)
        out.append(model.encoder(x))
    return (torch.cat(out, 0) if out else torch.empty(0)), valid

@torch.no_grad()
def pred_sim_strings(model, A, B, batch=512):
    model.eval(); sims = []
    for i in range(0, len(A), batch):
        xa = torch.stack([encode_pad(s) for s in A[i:i+batch]]).to(device)
        xb = torch.stack([encode_pad(s) for s in B[i:i+batch]]).to(device)
        d = torch.linalg.vector_norm(model.encoder(xa) - model.encoder(xb), dim=1)
        sims.append((1.0 - d / 2.0).cpu().numpy())
    return np.concatenate(sims) if sims else np.array([])

def feed_eval_pairs(feed):
    """All eval pairs scored in this alphabet with both proteins available."""
    sc, lk = SCORE_COL[feed], LOOKUP[feed]
    sub = eval_df.dropna(subset=[sc])
    return sub[sub['domain_a'].isin(lk) & sub['domain_b'].isin(lk)]

def feed_high_pairs(feed):
    """>= 0.70 eval pairs with both proteins in the retrieval pool (the queries)."""
    sc, lk = SCORE_COL[feed], LOOKUP[feed]
    inpool = set(all_domains) & set(lk)
    sub = eval_df.dropna(subset=[sc])
    return sub[(sub[sc] >= BAND_HIGH) & sub['domain_a'].isin(inpool) & sub['domain_b'].isin(inpool)]

def auroc_cell(model, feed):
    """Q1 - is-high AUROC. Returns (auroc, n_pos, n_total)."""
    sc, lk = SCORE_COL[feed], LOOKUP[feed]
    sub = feed_eval_pairs(feed)
    y = (sub[sc].values >= BAND_HIGH).astype(int)
    if y.sum() == 0 or y.sum() == len(y): return np.nan, int(y.sum()), len(y)
    pred = pred_sim_strings(model, [lk[d] for d in sub['domain_a']], [lk[d] for d in sub['domain_b']])
    return float(roc_auc_score(y, pred)), int(y.sum()), len(y)

## 8. Retrieval machinery — hits@10 (AA) and set-based frac@10 (SS, 3Di)

**Why two metrics.** In AA (20 letters) a query at `>= 0.70` has essentially one genuine
partner, so "is the designated partner in the top 10?" (**hits@10**) is fair. In SS (3
letters) and 3Di a query has *many* genuine `>= 0.70` neighbours, so we build the
**exact-Levenshtein neighbour set** (brute-force `normLev` of the query vs all 10,117 pool
proteins; the true set = everything `>= 0.70`) and report **frac@10 = retrieved / min(10,
|true set|)** = the fraction of the achievable top-10 high-sim neighbours the encoder
surfaces. `median_n_true` reports the crowding (how many genuine neighbours there are).

In [ ]:
def hits_at_k_designated(model, feed, k=K_RETRIEVAL):
    """Well-posed feeds (AA): rank the single designated partner. Returns (pct, n_q)."""
    sub = feed_high_pairs(feed); lk = LOOKUP[feed]
    pool_ids = [d for d in all_domains if d in lk]; idx = {d: i for i, d in enumerate(pool_ids)}
    pool_emb, _ = encode_pool(model, lk, pool_ids)
    hits = 0; nq = 0
    for a, b in sub[['domain_a', 'domain_b']].values:
        for q, partner in [(a, b), (b, a)]:
            if q not in idx or partner not in idx: continue
            qi = idx[q]
            esim = (1.0 - torch.linalg.vector_norm(pool_emb - pool_emb[qi], dim=1) / 2.0).cpu().numpy()
            esim[qi] = -np.inf
            rank = int(np.where(np.argsort(-esim) == idx[partner])[0][0]) + 1
            hits += int(rank <= k); nq += 1
    return (100.0 * hits / nq if nq else np.nan), nq

def build_oracle_cache(feed):
    """Exact-Levenshtein true neighbour sets (>= 0.70) per query. Crowded feeds only."""
    lk = LOOKUP[feed]
    pool_ids = [d for d in all_domains if d in lk]; idx = {d: i for i, d in enumerate(pool_ids)}
    pool_seqs = [lk[d] for d in pool_ids]; lens = np.array([len(s) for s in pool_seqs])
    sub = feed_high_pairs(feed)
    q_ids = sorted({d for pr in sub[['domain_a', 'domain_b']].values for d in pr})
    if not q_ids:
        return {'pool_ids': pool_ids, 'idx': idx, 'q_ids': [], 'true_sets': {}}
    D = rf_cdist([lk[q] for q in q_ids], pool_seqs, scorer=RFLevenshtein.distance, workers=-1).astype(float)
    true_sets = {}
    for r, q in enumerate(q_ids):
        qi = idx[q]; denom = np.maximum(lens, lens[qi]); denom[denom == 0] = 1
        osim = 1.0 - D[r] / denom; osim[qi] = -np.inf
        true_sets[q] = set(np.where(osim >= BAND_HIGH)[0].tolist())
    nt = [len(true_sets[q]) for q in q_ids]
    print(f'  oracle[{feed}]: {len(q_ids)} queries, median |true_set|={int(np.median(nt))}, max={max(nt)}')
    return {'pool_ids': pool_ids, 'idx': idx, 'q_ids': q_ids, 'true_sets': true_sets}

def frac_at_k_setbased(model, feed, cache, k=K_RETRIEVAL):
    """Crowded feeds (SS, 3Di): frac@k vs the exact-Lev true set. Returns (pct, n_q, median_n_true)."""
    lk = LOOKUP[feed]
    if not cache['q_ids']: return np.nan, 0, np.nan
    pool_emb, _ = encode_pool(model, lk, cache['pool_ids']); idx = cache['idx']
    fracs = []; nts = []
    for q in cache['q_ids']:
        qi = idx[q]; ts = cache['true_sets'][q]; nt = len(ts); nts.append(nt)
        if nt == 0: continue
        esim = (1.0 - torch.linalg.vector_norm(pool_emb - pool_emb[qi], dim=1) / 2.0).cpu().numpy()
        esim[qi] = -np.inf
        ret = len(set(np.argsort(-esim)[:k].tolist()) & ts)
        fracs.append(ret / min(k, nt))
    return (100.0 * float(np.mean(fracs)) if fracs else np.nan), len(fracs), float(np.median(nts))

# Oracle sets are encoder-independent -> build once for the crowded feeds.
print('Building exact-Levenshtein oracle neighbour sets (SS, 3Di)...')
ORACLE = {feed: build_oracle_cache(feed) for feed in ['SS', '3Di']}

## 9. Compute the 2x3 matrix — Accuracy (AUROC) + Retrieval (%)

In [ ]:
ENCODERS = [('AA-enc', model_aa), ('SS-enc', model_ss)]
FEEDS = ['AA', 'SS', '3Di']
RETR_METRIC = {'AA': 'hits@10', 'SS': 'frac@10', '3Di': 'frac@10'}

results = []
for enc_label, model in ENCODERS:
    for feed in FEEDS:
        auroc, n_pos, n_tot = auroc_cell(model, feed)
        if feed == 'AA':
            ret_pct, n_q = hits_at_k_designated(model, feed); med_true = 1.0
        else:
            ret_pct, n_q, med_true = frac_at_k_setbased(model, feed, ORACLE[feed])
        in_domain = ((enc_label, feed) in [('AA-enc', 'AA'), ('SS-enc', 'SS')])
        results.append({'encoder': enc_label, 'feed': feed,
                        'auroc': auroc, 'auroc_n_pos': n_pos, 'auroc_n_total': n_tot,
                        'retr_metric': RETR_METRIC[feed], 'retr_pct': ret_pct,
                        'retr_n_q': n_q, 'median_n_true': med_true,
                        'role': 'in-domain' if in_domain else 'cross-rep'}
        )
        print(f'{enc_label} x {feed:<4} | AUROC={auroc:.3f} (n_pos={n_pos}) | '
              f'{RETR_METRIC[feed]}={ret_pct:.0f}% (n_q={n_q}, med|true|={med_true:.0f})')

res_df = pd.DataFrame(results)

## 10. Chart 1 — Accuracy (AUROC is-high)

Grouped bars: x = feed modality, blue = AA-encoder, green = SS-encoder. Dashed line = 0.5
(chance). **AA bars are n_pos=5** (only 5 high-AA pairs exist in all of CATH) -> read AA as
underpowered; SS (n=333) and 3Di (n=38) are the trustworthy columns.

In [ ]:
def grouped_vals(metric):
    return ([next(r[metric] for r in results if r['encoder']=='AA-enc' and r['feed']==f) for f in FEEDS],
            [next(r[metric] for r in results if r['encoder']=='SS-enc' and r['feed']==f) for f in FEEDS])

aa_auroc, ss_auroc = grouped_vals('auroc')
x = np.arange(len(FEEDS)); w = 0.38
fig, ax = plt.subplots(figsize=(8.5, 5.2))
b1 = ax.bar(x - w/2, aa_auroc, w, label='AA-encoder', color=COL_AA_ENC)
b2 = ax.bar(x + w/2, ss_auroc, w, label='SS-encoder', color=COL_SS_ENC)
ax.axhline(0.5, ls='--', color='grey', lw=1)
ax.text(len(FEEDS)-0.5, 0.515, 'chance (0.5)', color='grey', fontsize=9, ha='right')
ax.bar_label(b1, fmt='%.3f', padding=2, fontsize=9)
ax.bar_label(b2, fmt='%.3f', padding=2, fontsize=9)
ax.set_ylim(0, 1.05); ax.set_yticks(np.arange(0, 1.01, 0.2))
ax.set_ylabel('AUROC  (is-high ≥ 0.70)')
ax.set_xlabel('Feed modality')
ax.set_xticks(x); ax.set_xticklabels(FEEDS)
ax.set_title('Accuracy: can embedding distance separate a high-similarity pair from a random one?')
# n_pos caveat under each group
for i, f in enumerate(FEEDS):
    npos = next(r['auroc_n_pos'] for r in results if r['feed']==f)
    ax.annotate(f'n_pos={npos}', (i, 0), (0, -28), textcoords='offset points',
                ha='center', va='top', fontsize=8, color='dimgray')
ax.legend(loc='lower left', framealpha=0.9)
plt.tight_layout(); plt.savefig('colab22_accuracy_auroc.png', dpi=150, bbox_inches='tight'); plt.show()

## 11. Chart 2 — Retrieval (%)

Grouped bars: x = feed modality, blue = AA-encoder, green = SS-encoder. **AA uses hits@10**
(designated partner), **SS & 3Di use frac@10** (set-based, vs the exact-Levenshtein
neighbour set). `med|true|` under SS/3Di = how crowded the pool is for that feed (median
number of genuine `>= 0.70` neighbours per query).

In [ ]:
aa_retr, ss_retr = grouped_vals('retr_pct')
x = np.arange(len(FEEDS)); w = 0.38
fig, ax = plt.subplots(figsize=(8.5, 5.2))
b1 = ax.bar(x - w/2, aa_retr, w, label='AA-encoder', color=COL_AA_ENC)
b2 = ax.bar(x + w/2, ss_retr, w, label='SS-encoder', color=COL_SS_ENC)
ax.bar_label(b1, fmt='%.0f%%', padding=2, fontsize=9)
ax.bar_label(b2, fmt='%.0f%%', padding=2, fontsize=9)
ax.set_ylim(0, 110); ax.set_yticks(np.arange(0, 101, 20))
ax.set_ylabel('Retrieval rate (%)')
ax.set_xlabel('Feed modality')
ax.set_xticks(x)
ax.set_xticklabels([f'{f}\n({RETR_METRIC[f]})' for f in FEEDS])
ax.set_title('Retrieval: can it find high-similarity neighbours in a crowded pool?')
for i, f in enumerate(FEEDS):
    if f == 'AA': continue
    mt = next(r['median_n_true'] for r in results if r['feed']==f)
    ax.annotate(f'med|true|={mt:.0f}', (i, 0), (0, -28), textcoords='offset points',
                ha='center', va='top', fontsize=8, color='dimgray')
ax.legend(loc='upper right', framealpha=0.9)
plt.tight_layout(); plt.savefig('colab22_retrieval.png', dpi=150, bbox_inches='tight'); plt.show()

## 12. Summary table + CSV export

In [ ]:
print('=' * 92)
print('COLAB22 SUMMARY — 2x3 cross-representation matrix (encoder x feed)')
print('=' * 92)
print(f'{"encoder":<9}{"feed":<6}{"role":<11}{"AUROC":>7}{"n_pos":>7}'
      f'{"retr metric":>13}{"retr %":>9}{"med|true|":>11}')
print('-' * 92)
for r in results:
    print(f"{r['encoder']:<9}{r['feed']:<6}{r['role']:<11}{r['auroc']:>7.3f}{r['auroc_n_pos']:>7}"
          f"{r['retr_metric']:>13}{r['retr_pct']:>8.0f}%{r['median_n_true']:>11.0f}")

res_df.to_csv('colab22_summary.csv', index=False)
print('\nSaved: colab22_summary.csv, colab22_accuracy_auroc.png, colab22_retrieval.png')

## How to read this notebook

**Two questions, one matrix.** Each (encoder, feed) cell carries both an **accuracy**
number (AUROC is-high) and a **retrieval** number (hits@10 for AA, frac@10 for SS/3Di).

- **Diagonal = in-domain** (AA-enc x AA, SS-enc x SS): the encoder was trained on that
  alphabet. **Off-diagonal = cross-representation** (zero-shot transfer). **3Di has no
  in-domain bar** - we never trained a 3Di encoder, so both 3Di bars are cross-rep.
- **Accuracy (AUROC)** is the strong, honest number: high-sim pairs are well-separated from
  random pairs across modalities. AA is n_pos=5 (underpowered - lean on SS/3Di).
- **Retrieval** splits by task well-posedness: AA hits@10 is high because the partner is
  unique; SS/3Di frac@10 is bounded by **crowding** (`med|true|`) - in a low-entropy
  alphabet a query has many equally-valid neighbours, so single-target retrieval is
  intrinsically harder. frac@10 measures the fair thing: of the achievable high-sim
  neighbours, how many did we surface.

The split between a strong AUROC and a modest SS/3Di frac@10 is the cross-rep story:
the encoder *can tell* high-sim from random (discrimination transfers), but *pinpointing*
one neighbour in a crowded low-entropy pool is an ill-posed retrieval task, not an encoder
failure.